In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from typing import Dict, List, Tuple
from data_loading import *
from loss_funcs import *

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}") # I have a 3090
    torch.set_float32_matmul_precision("medium")
    print("float32 matmul precision set to 'medium'")

# ── Moirai / training-stack imports ──────────────────────────────────────────
# pip install "uni2ts @ git+https://github.com/SalesforceAIResearch/uni2ts.git" gluonts mlflow
# Requires Python >= 3.10 (uni2ts 2.x)
import os
import copy
from tqdm.notebook import tqdm

import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from gluonts.dataset.common import ListDataset

# uni2ts 2.x: MoiraiModule owns from_pretrained; MoiraiForecast/MoiraiFinetune take module=
from uni2ts.model.moirai import MoiraiForecast, MoiraiFinetune
from uni2ts.model.moirai.module import MoiraiModule

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, TQDMProgressBar

import mlflow
import mlflow.pytorch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device           : {DEVICE}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Column lists — multivariate Moirai feature spec
# ─────────────────────────────────────────────────────────────────────────────
# Each column gets one of three roles for Moirai:
#   • target                   — the variable we forecast (Y_COL)
#   • feat_dynamic_real        — known across context AND prediction horizon
#                                (weather forecasts, calendar, capacity, DAM price)
#   • past_feat_dynamic_real   — only known historically; values during the
#                                prediction window are unknown to the model
#                                (balancing prices, which settle after the fact)
#
# Static columns identify the series (per-station constants); we use them to
# split / group windows but do NOT feed them as input variates because Moirai
# 1.1-R has no native static-feature head. eic_code stays in GROUP_COL.
# Calendar columns are integer-coded already and are passed as numeric reals.

weather_cols_to_keep = [
    "temperature_2m",
    "precipitation",
    "cloud_cover",
    "wind_speed_10m",
    "shortwave_radiation",
    "direct_normal_irradiance",
]

other_cols = [          # not static
    "dam_price", "buy_bm_price", "sell_bm_price",
    "max_power", "max_solar", "max_ev",
]

cat_columns = [
    "eic_code", "dso_desc", "station_type", "oblast",
    "Month", "Day", "Hour", "day_of_week", "season",
]

static_cols = [
    "latitude", "longitude", "eic_code", "dso_desc", "station_type", "oblast",
]

calendar_cols = ["Month", "Day", "Hour", "day_of_week", "season"]

time_cols = ["datetime", "time_idx"]

# ─────────────────────────────────────────────────────────────────────────────
# Role assignment for the multivariate Moirai input
# ─────────────────────────────────────────────────────────────────────────────
# Known at prediction time → feat_dynamic_real (full window of length seq_len)
FUTURE_KNOWN_COLS = (
    weather_cols_to_keep                   # weather forecasts are available
    + calendar_cols                        # deterministic from the calendar
    + ["max_power", "max_solar", "max_ev"] # capacity ceilings, known a priori
    + ["dam_price"]                        # day-ahead price is known the day before
)

# Only known historically → past_feat_dynamic_real (context length only)
PAST_ONLY_COLS = ["buy_bm_price", "sell_bm_price"]

# Required by money_pct loss / inference-side reattachment (NOT model inputs).
# Note: dam_price is also a model input above, but we still need the raw column
# alongside the prediction frame to compute money_pct downstream.
PRICE_COLS_FOR_LOSS = ["dam_price", "buy_bm_price", "sell_bm_price"]

print(f"y col            : {Y_COL}")
print(f"group col        : {GROUP_COL}")
print(f"future-known     : {len(FUTURE_KNOWN_COLS)}  → feat_dynamic_real")
print(f"past-only        : {len(PAST_ONLY_COLS)}  → past_feat_dynamic_real")
print(f"target variates  : 1  (Y_COL)")
print(f"total variates   : {1 + len(FUTURE_KNOWN_COLS) + len(PAST_ONLY_COLS)}")
print()
print("future-known cols:", FUTURE_KNOWN_COLS)
print("past-only    cols:", PAST_ONLY_COLS)

In [ ]:
# this data has a date column and categorical columns are kept intact and will need to be handled.
# "time_idx" is already built in train, val and test and is continuous through them
print("Loading train …")
train = load_and_prepare(TRAIN_PATH_WITH_DATETME)

print("Loading val   …")
val = load_and_prepare(VAL_PATH_WITH_DATETME)

print("Loading test  …")
test = load_and_prepare(TEST_PATH_WITH_DATETME)

print(f"train: {train.shape}")
print(f"val : {val.shape}")
print(f"test: {test.shape}")

In [ ]:
# # If a model cant handle categorical columns natively, or through embeddings use this data
# # It has no datetime column and all columns are numeric, as all cat column were ohe
# # GROUP_COL is the only exception, and is not ohe. Ohe it before training
# print("Loading train …")
# train = load_and_prepare(TRAIN_PATH_OHE)
#
# print("Loading val   …")
# val = load_and_prepare(VAL_PATH_OHE)
#
# print("Loading test  …")
# test = load_and_prepare(TEST_PATH_OHE)
#
# print(f"train: {train.shape}")
# print(f"val  : {val.shape}")
# print(f"test : {test.shape}")

In [ ]:
# this cell samples locations, I'll use it if training takes too long, otherwise don't touch it

TARGET_STATIONS = 395

station_stats = (
    train.groupby(GROUP_COL)
    .agg(rows=(Y_COL, "count"))
    .reset_index()
    .sort_values("rows", ascending=False)
)

sampled_stations = station_stats.sample(
    n=TARGET_STATIONS, random_state=42
)[GROUP_COL].values

print(f"Stations: {len(sampled_stations)}")

train = train[train[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
val   = val[val[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
test  = test[test[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)

print(f"Train rows : {len(train):,}")
print(f"Val rows   : {len(val):,}")
print(f"Test rows  : {len(test):,}")

In [ ]:
training_cutoff = train["time_idx"].max()
val_cutoff      = val["time_idx"].max()
test_cutoff     = test["time_idx"].max()

print(f"training cutoff : {training_cutoff}")
print(f"val cutoff      : {val_cutoff}")
print(f"test cutoff     : {test_cutoff}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Moirai config — sizing and training hyperparameters (multivariate)
# ─────────────────────────────────────────────────────────────────────────────
# Moirai is a *foundation* time-series model (Salesforce, ICML 2024). It works
# in patch space: each variate's window of length CONTEXT_LEN + PRED_LEN is
# reshaped into (n_patches, patch_size). Multiple variates are stacked along
# the patch dim and distinguished by `variate_id`. PATCH_SIZE must divide
# CONTEXT_LEN + PRED_LEN.

MODEL_ID    = "Salesforce/moirai-1.1-R-small"   # small=91M / base=236M / large=311M
CONTEXT_LEN = 168    # 1-week look-back (hourly)
PRED_LEN    = 48     # 48-hour forecast horizon (matches val/test rolling)
PATCH_SIZE  = 8      # (168 + 48) / 8 = 27 patches per variate
NUM_SAMPLES = 100    # Monte-Carlo samples for the predictive distribution at inference

# ── Multivariate sizing ──────────────────────────────────────────────────────
# Moirai counts each numeric column as a "variate". The target is variate 0;
# feat_dynamic_real and past_feat_dynamic_real columns each get their own
# variate id. max_dim is the largest variate count the finetuner will expect
# at runtime — it must be ≥ the actual variate count we feed.
FEAT_DYNAMIC_REAL_DIM      = len(FUTURE_KNOWN_COLS)   # known across full window
PAST_FEAT_DYNAMIC_REAL_DIM = len(PAST_ONLY_COLS)      # known only for context
TOTAL_VARIATES             = 1 + FEAT_DYNAMIC_REAL_DIM + PAST_FEAT_DYNAMIC_REAL_DIM
MAX_DIM                    = max(TOTAL_VARIATES, 128) # head room for AddVariateIndex

# Fine-tuning
TRAIN_STRIDE = 24    # one window per day — fast, diverse coverage
BATCH_SIZE   = 32    # halved vs univariate run: ~K× more patches per sample
EPOCHS       = 10
LR           = 1e-4
WEIGHT_DECAY = 1e-2
SMAPE_WEIGHT = 0.1   # SMAPE regulariser in the money_pct training loss
NUM_WORKERS  = 0     # 0 on Windows / Jupyter (avoids multiprocessing deadlock)
SEED         = 42

CKPT_DIR  = "../checkpoints"
SAVE_PATH = os.path.join(CKPT_DIR, "moirai_finetuned_money_mv.pt")
os.makedirs(CKPT_DIR, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Total variates per sample : {TOTAL_VARIATES}  "
      f"(target=1, future-known={FEAT_DYNAMIC_REAL_DIM}, past-only={PAST_FEAT_DYNAMIC_REAL_DIM})")
print(f"Patches per variate       : {(CONTEXT_LEN + PRED_LEN) // PATCH_SIZE}  "
      f"({PRED_LEN // PATCH_SIZE} prediction patches)")
print(f"Total patches per sample  : {TOTAL_VARIATES * (CONTEXT_LEN + PRED_LEN) // PATCH_SIZE}")

## Moirai dataset preparation (multivariate)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Multivariate Moirai dataset preparation
# ─────────────────────────────────────────────────────────────────────────────
# Moirai (uni2ts v2.x) consumes data in PATCH form. For a multivariate sample
# with K variates and P patches per variate, every tensor has a leading
# patch-dim of size K*P:
#
#   target          : (K*P, MAX_PATCH=128) float — values, zero-padded after patch_size
#   observed_mask   : (K*P, MAX_PATCH)     bool  — True where data is real & known
#   prediction_mask : (K*P,)               bool  — True ONLY on the TARGET
#                                                  variate's prediction patches
#                                                  (loss is only on the target)
#   time_id         : (K*P,)               long  — patch position 0..P-1, repeated per variate
#   variate_id      : (K*P,)               long  — 0 for target, 1..K-1 for covariates
#   patch_size      : (K*P,)               long  — patch_size used for this sample
#   sample_id       : (K*P,)               long  — segment id (single segment → 0)
#
# Variate layout (top → bottom rows):
#   0                        target (Y_COL)
#   1 .. F                   future-known feat_dynamic_real (FUTURE_KNOWN_COLS)
#   F+1 .. F+P_pst           past-only past_feat_dynamic_real (PAST_ONLY_COLS)
# where F = FEAT_DYNAMIC_REAL_DIM, P_pst = PAST_FEAT_DYNAMIC_REAL_DIM.
#
# observed_mask semantics per role:
#   • target               : True everywhere we have data (NaNs → False)
#   • feat_dynamic_real    : True over the full seq_len (values known throughout)
#   • past_feat_dynamic_real: True ONLY on the context portion;
#                             prediction-horizon patches → False (unknown values)
#
# prediction_mask is True ONLY on the target's prediction patches. The loss is
# computed against those positions; covariates are conditioning information.
#
# Per-variate normalization: each variate is z-scored using its OWN context
# window's mean/std. The target's mu/sig are stored separately so the
# money_pct loss can de-normalize predictions back into kWh.
#
# Static columns (latitude, longitude, eic_code, dso_desc, station_type,
# oblast) are NOT injected as variates because Moirai 1.1-R has no static-
# feature head. Per-station behaviour is captured implicitly: each window
# comes from a single eic_code, and the per-window normalization absorbs
# magnitude differences across stations.

# ─────────────────────────────────────────────────────────────────────────────
# Build per-station numpy arrays. We carry one array per role + the price
# triplet needed by the money loss.
# ─────────────────────────────────────────────────────────────────────────────
def build_series_dict_mv(df: pd.DataFrame) -> Dict[str, dict]:
    out = {}
    for eic, grp in df.groupby(GROUP_COL):
        grp = grp.sort_values("time_idx")
        out[eic] = {
            "values":      grp[Y_COL].values.astype(np.float32),
            "time_idx":    grp["time_idx"].values.astype(np.int64),
            "datetime":    grp["datetime"].values,
            # (T, F)  future-known covariates (used as model input)
            "feat_future": grp[FUTURE_KNOWN_COLS].to_numpy(dtype=np.float32),
            # (T, P)  past-only covariates (used as model input)
            "feat_past":   grp[PAST_ONLY_COLS].to_numpy(dtype=np.float32),
            # raw price triplet used by the money loss (not a model input itself,
            # though dam_price is *also* a model input via FUTURE_KNOWN_COLS)
            "dam_price":     grp["dam_price"].values.astype(np.float32),
            "sell_bm_price": grp["sell_bm_price"].values.astype(np.float32),
            "buy_bm_price":  grp["buy_bm_price"].values.astype(np.float32),
        }
    return out


def fill_nan_1d(arr: np.ndarray) -> np.ndarray:
    if not np.isnan(arr).any():
        return arr
    return (
        pd.Series(arr)
        .interpolate(method="linear", limit_direction="both")
        .values.astype(np.float32)
    )


def fill_nan_2d(arr: np.ndarray) -> np.ndarray:
    """Column-wise linear interpolation for an (T, K) array."""
    if not np.isnan(arr).any():
        return arr
    df = pd.DataFrame(arr).interpolate(method="linear", limit_direction="both")
    # Any column that was entirely NaN stays NaN — fill those with 0.
    return df.fillna(0.0).values.astype(np.float32)


# Continuous full-history view for rolling inference (val + test windows need
# train-side context, so we concat). time_idx is already continuous across
# splits per the project setup.
full_df = (
    pd.concat([train, val, test], ignore_index=True)
    .sort_values([GROUP_COL, "time_idx"])
    .reset_index(drop=True)
)

train_series = build_series_dict_mv(train)
full_series  = build_series_dict_mv(full_df)

lengths = [len(v["values"]) for v in train_series.values()]
print(f"Stations         : {len(train_series)}")
print(f"Train series len : min={min(lengths)}, max={max(lengths)}, mean={np.mean(lengths):.0f}")
print(f"feat_future shape per station : (T, {FEAT_DYNAMIC_REAL_DIM})")
print(f"feat_past   shape per station : (T, {PAST_FEAT_DYNAMIC_REAL_DIM})")


# ─────────────────────────────────────────────────────────────────────────────
# Sliding-window dataset that emits multivariate Moirai patches.
# ─────────────────────────────────────────────────────────────────────────────
class MoiraiMVWindowDataset(Dataset):
    """Sliding-window dataset producing multivariate Moirai patch tensors.

    Each emitted sample stacks K = 1 + F + P_pst variates along the patch dim:
        rows 0          → target
        rows 1..F       → future-known feat_dynamic_real
        rows F+1..K-1   → past-only past_feat_dynamic_real

    The prediction loss is computed only on the target's prediction patches —
    `prediction_mask` is True there and False everywhere else.

    Per-window auxiliary tensors carried alongside the model inputs:
        dam_price, sell_bm_price, buy_bm_price : (P, MAX_PATCH) — raw prices
            (per-hour) for the money_pct loss
        norm_mu, norm_sig                       : scalars — TARGET's z-score
            stats used to invert normalization in the loss
    """

    MAX_PATCH = 128  # moirai-1.1-R fixed max patch size — pad shorter patches to this width

    def __init__(self, series_dict, context_len, pred_len, stride, patch_size,
                 holdout_tail: int = 0, tail_only: bool = False):
        self.context_len    = context_len
        self.pred_len       = pred_len
        self.seq_len        = context_len + pred_len
        self.patch_size     = patch_size
        self.n_patches      = self.seq_len // patch_size
        self.n_pred_patches = pred_len // patch_size
        self.n_ctx_patches  = self.n_patches - self.n_pred_patches

        # Each entry stores the slice indices into the per-station arrays so
        # we can re-extract everything lazily in __getitem__ without bloating RAM.
        # Tuple: (eic, start_idx)
        self.windows: List[Tuple[str, int]] = []
        self.series  = series_dict

        for eic, data in series_dict.items():
            T = len(data["values"])
            if T < self.seq_len:
                continue

            if tail_only:
                # Single window covering the last seq_len hours of this station.
                self.windows.append((eic, T - self.seq_len))
            else:
                # Sliding training windows; stop before the holdout tail so
                # validation hours never appear inside any training window.
                last_start = T - self.seq_len - holdout_tail
                if last_start < 0:
                    continue
                for s in range(0, last_start + 1, stride):
                    self.windows.append((eic, s))

    def __len__(self):
        return len(self.windows)

    # ─────────────────────────────────────────────────────────────────────
    # Helpers
    # ─────────────────────────────────────────────────────────────────────
    def _zpatch_1d(self, seq: np.ndarray) -> torch.Tensor:
        """(seq_len,) → (n_patches, MAX_PATCH) zero-padded after patch_size."""
        P, ps = self.n_patches, self.patch_size
        t = torch.zeros(P, self.MAX_PATCH, dtype=torch.float32)
        t[:, :ps] = torch.from_numpy(seq.reshape(P, ps).astype(np.float32))
        return t

    def _omask_full(self, valid: np.ndarray) -> torch.Tensor:
        """`valid` shape (seq_len,) bool → (n_patches, MAX_PATCH) observed_mask."""
        P, ps = self.n_patches, self.patch_size
        m = torch.zeros(P, self.MAX_PATCH, dtype=torch.bool)
        m[:, :ps] = torch.from_numpy(valid.reshape(P, ps))
        return m

    # ─────────────────────────────────────────────────────────────────────
    # Sample builder
    # ─────────────────────────────────────────────────────────────────────
    def __getitem__(self, idx):
        eic, s = self.windows[idx]
        e = s + self.seq_len
        data = self.series[eic]

        # ── Pull raw slices ─────────────────────────────────────────────
        target_seq   = fill_nan_1d(data["values"][s:e].copy())                  # (seq_len,)
        feat_future  = fill_nan_2d(data["feat_future"][s:e, :].copy())          # (seq_len, F)
        feat_past    = fill_nan_2d(data["feat_past"][s:e, :].copy())            # (seq_len, P_pst)
        dam_seq      = fill_nan_1d(data["dam_price"][s:e].copy())               # (seq_len,)
        sell_seq     = fill_nan_1d(data["sell_bm_price"][s:e].copy())
        buy_seq      = fill_nan_1d(data["buy_bm_price"][s:e].copy())

        F   = FEAT_DYNAMIC_REAL_DIM
        Ppt = PAST_FEAT_DYNAMIC_REAL_DIM
        K   = 1 + F + Ppt
        P   = self.n_patches
        ps  = self.patch_size
        Cp  = self.n_ctx_patches

        # ── Per-variate z-score using context-window stats only ─────────
        # Target stats are stored separately because the loss needs them
        # to invert normalization.
        ctx_t = target_seq[: self.context_len]
        mu_t  = float(ctx_t.mean())
        sig_t = max(float(ctx_t.std()), 1e-5)
        target_n = (target_seq - mu_t) / sig_t

        # Future-known covariates: normalize using their own context stats.
        # Note: their prediction-horizon values ARE used as input, so we
        # normalize the WHOLE seq_len with stats taken only from the context
        # region (same convention as the target).
        if F > 0:
            ctx_f = feat_future[: self.context_len, :]                          # (CTX, F)
            mu_f  = ctx_f.mean(axis=0)                                          # (F,)
            sig_f = np.maximum(ctx_f.std(axis=0), 1e-5)                         # (F,)
            feat_future_n = (feat_future - mu_f) / sig_f                        # (seq_len, F)
        else:
            feat_future_n = feat_future  # empty

        # Past-only covariates: normalize the WHOLE seq_len with context stats.
        # Prediction-horizon positions will be masked out of observed_mask
        # below, so their actual numeric value doesn't matter — but we still
        # normalize them defensively.
        if Ppt > 0:
            ctx_p = feat_past[: self.context_len, :]
            mu_p  = ctx_p.mean(axis=0)
            sig_p = np.maximum(ctx_p.std(axis=0), 1e-5)
            feat_past_n = (feat_past - mu_p) / sig_p
        else:
            feat_past_n = feat_past

        # ── Build patch tensors per variate, then concat along patch dim ──
        target_patches = self._zpatch_1d(target_n)                              # (P, MAX_PATCH)
        target_omask   = self._omask_full(np.ones(self.seq_len, dtype=bool))    # (P, MAX_PATCH)

        # feat_future: observed everywhere
        ff_patches_list, ff_omask_list = [], []
        for k in range(F):
            ff_patches_list.append(self._zpatch_1d(feat_future_n[:, k]))
            ff_omask_list.append(self._omask_full(np.ones(self.seq_len, dtype=bool)))

        # feat_past: observed only on the context portion
        past_valid = np.zeros(self.seq_len, dtype=bool)
        past_valid[: self.context_len] = True
        fp_patches_list, fp_omask_list = [], []
        for k in range(Ppt):
            fp_patches_list.append(self._zpatch_1d(feat_past_n[:, k]))
            fp_omask_list.append(self._omask_full(past_valid))

        # Concat along the patch dim → (K*P, MAX_PATCH)
        all_patches = [target_patches] + ff_patches_list + fp_patches_list
        all_omasks  = [target_omask]   + ff_omask_list   + fp_omask_list
        target = torch.cat(all_patches, dim=0)                                  # (K*P, MAX_PATCH)
        omask  = torch.cat(all_omasks,  dim=0)                                  # (K*P, MAX_PATCH)

        # prediction_mask: True only on target's prediction patches
        pred_mask = torch.zeros(K * P, dtype=torch.bool)
        pred_mask[P - self.n_pred_patches : P] = True   # last `n_pred_patches`
                                                        # of the TARGET block
        # variate_id: 0..K-1, each repeated P times
        variate_id = torch.repeat_interleave(
            torch.arange(K, dtype=torch.long), repeats=P
        )
        # time_id: 0..P-1, repeated K times
        time_id = torch.arange(P, dtype=torch.long).repeat(K)
        # patch_size: constant
        patch_size_t = torch.full((K * P,), ps, dtype=torch.long)
        # sample_id: single segment
        sample_id = torch.zeros(K * P, dtype=torch.long)

        # ── Auxiliary tensors used by money_pct in the loss ─────────────
        # Prices are kept in their RAW per-MWh form. They sit on the target's
        # patch grid (P patches), NOT the multivariate K*P grid, since the
        # loss is computed only on the target.
        def _to_patch_1d(arr):
            t = torch.zeros(P, self.MAX_PATCH, dtype=torch.float32)
            t[:, :ps] = torch.from_numpy(arr.reshape(P, ps).astype(np.float32))
            return t

        return {
            # Moirai forward-pass inputs
            "target":          target,                    # (K*P, MAX_PATCH)
            "observed_mask":   omask,                     # (K*P, MAX_PATCH)
            "prediction_mask": pred_mask,                 # (K*P,)
            "time_id":         time_id,                   # (K*P,)
            "variate_id":      variate_id,                # (K*P,)
            "patch_size":      patch_size_t,              # (K*P,)
            "sample_id":       sample_id,                 # (K*P,)
            # Money-loss auxiliaries (target grid only)
            "dam_price":       _to_patch_1d(dam_seq),     # (P, MAX_PATCH)
            "sell_bm_price":   _to_patch_1d(sell_seq),
            "buy_bm_price":    _to_patch_1d(buy_seq),
            "norm_mu":         torch.tensor(mu_t,  dtype=torch.float32),
            "norm_sig":        torch.tensor(sig_t, dtype=torch.float32),
        }


def collate_moirai(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}


train_ds = MoiraiMVWindowDataset(
    train_series, CONTEXT_LEN, PRED_LEN, TRAIN_STRIDE, PATCH_SIZE,
    holdout_tail=CONTEXT_LEN + PRED_LEN,   # reserve the last seq_len hours per station
)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_moirai, num_workers=NUM_WORKERS, pin_memory=True,
)

# Held-out per-epoch loss-eval set: the last seq_len hours of each station's
# training history. Used ONLY for the per-epoch print of money_pct / smape on
# unseen data — the project's val/test splits remain reserved for the rolling
# inference cells below.
val_loss_ds = MoiraiMVWindowDataset(
    train_series, CONTEXT_LEN, PRED_LEN, TRAIN_STRIDE, PATCH_SIZE,
    tail_only=True,
)
val_loss_loader = DataLoader(
    val_loss_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_moirai, num_workers=NUM_WORKERS, pin_memory=True,
)
print(f"Training windows : {len(train_ds):,}")
print(f"Val-loss windows : {len(val_loss_ds):,}  (one tail window per station)")
print(f"Batches / epoch  : {len(train_loader):,}")
print(f"Patches / sample : {train_ds.n_patches} per variate, "
      f"{TOTAL_VARIATES * train_ds.n_patches} total "
      f"(pred patches on target: {train_ds.n_pred_patches})")

## Money-loss wrapper (differentiable PyTorch)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Differentiable PyTorch wrappers around money_pct and SMAPE
# ─────────────────────────────────────────────────────────────────────────────
# Mirrors loss_funcs.money_pct exactly:
#   • prices are per-MWh → divided by 1000 before multiplying with kWh
#   • NO tolerance band — every kWh of deviation is penalised
#   • over-prediction (diff > 0): rebate at sell_bm_price on the excess
#   • under-prediction (diff < 0): pay buy_bm_price for the deficit
#   • result is POOLED: sum(real_spend − min_spend) / (sum(min_spend) / 100)
# Pooled aggregation means a single value per batch, NOT a mean of per-sample
# percentages. This matches your project's reference and keeps train/val/test
# numbers directly comparable to the rolling_eval cell.

def money_pct_torch(
    y_true: torch.Tensor,         # (N,) — actual kWh
    y_pred: torch.Tensor,         # (N,) — predicted kWh
    dam_price: torch.Tensor,      # (N,) — day-ahead price (per MWh)
    sell_bm_price: torch.Tensor,  # (N,) — balancing sell-back price (per MWh)
    buy_bm_price: torch.Tensor,   # (N,) — balancing buy price (per MWh)
    eps: float = 1e-6,
) -> torch.Tensor:
    p_dam  = dam_price     / 1000.0
    p_sell = sell_bm_price / 1000.0
    p_buy  = buy_bm_price  / 1000.0

    min_spend      = y_true * p_dam
    ordered_amount = y_pred * p_dam
    diff           = y_pred - y_true

    zero       = torch.zeros_like(diff)
    real_spend = (
        ordered_amount
        - torch.where(diff > 0, p_sell * diff, zero)
        - torch.where(diff < 0, p_buy  * diff, zero)
    )
    return (real_spend - min_spend).sum() / min_spend.sum().clamp_min(eps) * 100.0


def smape_torch(
    y_true: torch.Tensor,
    y_pred: torch.Tensor,
    eps: float = 1e-6,
) -> torch.Tensor:
    """Symmetric MAPE on real kWh, scale-invariant by construction."""
    denom = torch.clamp((y_true.abs() + y_pred.abs()) / 2.0, min=eps)
    return (torch.abs(y_pred - y_true) / denom).mean() * 100.0


# ─────────────────────────────────────────────────────────────────────────────
# Parity check: confirm money_pct_torch agrees with loss_funcs.money_pct
# ─────────────────────────────────────────────────────────────────────────────
_rng = np.random.default_rng(0)
_n = 4096
_y_true_np = _rng.uniform(10.0, 200.0, size=_n).astype(np.float32)
_err_pct   = _rng.normal(0.0, 0.20, size=_n).astype(np.float32)
_y_pred_np = (_y_true_np * (1.0 + _err_pct)).clip(min=0.1)
_dam_np    = _rng.uniform(1500.0, 4500.0, size=_n).astype(np.float32)
_sell_np   = (_dam_np * 0.6).astype(np.float32)
_buy_np    = (_dam_np * 1.4).astype(np.float32)

_torch_v = float(money_pct_torch(
    torch.from_numpy(_y_true_np), torch.from_numpy(_y_pred_np),
    torch.from_numpy(_dam_np),   torch.from_numpy(_sell_np), torch.from_numpy(_buy_np),
).item())
_numpy_v = float(money_pct(_y_true_np, _y_pred_np, _dam_np, _sell_np, _buy_np))

print("── money_pct parity check ───────────────────────────────────")
print(f"  money_pct_torch (PyTorch) : {_torch_v:.6f}")
print(f"  money_pct       (numpy)   : {_numpy_v:.6f}")
print(f"  abs delta                 : {abs(_torch_v - _numpy_v):.6e}")
assert abs(_torch_v - _numpy_v) < 1e-3, "money_pct_torch diverges from loss_funcs.money_pct"
print("  → match ✓")

# Keys our dataset adds that MoiraiModule.forward should NOT see.
_MOIRAI_FORWARD_KEYS = {
    "target", "observed_mask", "sample_id", "time_id", "variate_id",
    "prediction_mask", "patch_size",
}


# ─────────────────────────────────────────────────────────────────────────────
# Custom MoiraiFinetune that swaps NLL for money_pct on the predicted point
# forecast (mean of the predictive distribution), using PER-HOUR REAL PRICES
# from the input batch — not global averages.
# ─────────────────────────────────────────────────────────────────────────────
# Multivariate detail: prediction_mask is True only on the TARGET variate's
# prediction patches. distr.mean[combined] therefore extracts only those
# target positions — covariate variates contribute conditioning but never
# enter the loss.
#
# Critical detail — the model operates in NORMALIZED space (per-window
# z-scoring). To compute money_pct in real currency we must un-normalize both
# the prediction and the target back to kWh using the per-window mu/sig the
# dataset stored. Without this, the loss would be scaled by sig and
# meaningless across windows with different consumption magnitudes.

class MoneyLossMoiraiFinetune(MoiraiFinetune):
    """Replaces PackedNLLLoss with money_pct + smape on the TARGET-variate
    distribution mean, using per-hour real prices carried by the dataset.
    """

    def __init__(self, *args, smape_weight: float = 0.1,
                 target_n_patches: int = 27, **kwargs):
        # target_n_patches = (CONTEXT_LEN + PRED_LEN) // PATCH_SIZE — the number
        # of patches on the TARGET variate. Used to slice price tensors (which
        # live on the target grid) onto the same positions that prediction_mask
        # selects from the multivariate (K*P) grid.
        super().__init__(*args, **kwargs)
        self.smape_weight     = smape_weight
        self.target_n_patches = target_n_patches

    # ─────────────────────────────────────────────────────────────────────
    # Shared helper: forward-pass + de-normalise + flatten prediction-step
    # tensors. Returns everything both training_step and validation_step
    # need to compute money_pct and smape on real kWh.
    # ─────────────────────────────────────────────────────────────────────
    def _forward_and_flatten(self, batch):
        moirai_inputs = {k: v for k, v in batch.items() if k in _MOIRAI_FORWARD_KEYS}
        distr = self.module(**moirai_inputs)

        target = batch["target"]                 # (B, K*P, MAX_PATCH), normalized
        pmask  = batch["prediction_mask"]        # (B, K*P)
        omask  = batch["observed_mask"]          # (B, K*P, MAX_PATCH)
        combined = pmask.unsqueeze(-1) & omask   # (B, K*P, MAX_PATCH)

        # `combined` is True ONLY on target prediction patches (because
        # prediction_mask was constructed to be True only there). So both
        # tensors below are flat sequences of target-prediction-step values.
        y_pred_norm = distr.mean[combined]
        y_true_norm = target[combined]

        # Broadcast TARGET-variate mu/sig to (B, K*P, MAX_PATCH) and flatten.
        # Same scalar broadcasts cleanly across the K*P axis — only target
        # rows are selected anyway via `combined`.
        mu_b  = batch["norm_mu"].view(-1, 1, 1).expand_as(target)
        sig_b = batch["norm_sig"].view(-1, 1, 1).expand_as(target)
        mu_flat  = mu_b[combined]
        sig_flat = sig_b[combined]

        y_pred_kwh = y_pred_norm * sig_flat + mu_flat
        y_true_kwh = y_true_norm * sig_flat + mu_flat

        # Prices live on the target grid: (B, P, MAX_PATCH). The TARGET block
        # of `combined` covers rows [0..P) of the K*P axis, which exactly
        # corresponds to the price grid. Slice to the target block, then index
        # with the same predicate.
        Pt = self.target_n_patches
        target_block_combined = combined[:, :Pt, :]   # (B, P, MAX_PATCH)
        dam  = batch["dam_price"][target_block_combined]
        sell = batch["sell_bm_price"][target_block_combined]
        buy  = batch["buy_bm_price"][target_block_combined]

        return y_true_kwh, y_pred_kwh, dam, sell, buy

    def training_step(self, batch, batch_idx):
        y_true_kwh, y_pred_kwh, dam, sell, buy = self._forward_and_flatten(batch)

        money_loss = money_pct_torch(y_true_kwh, y_pred_kwh, dam, sell, buy)
        smape_pct  = smape_torch(y_true_kwh, y_pred_kwh)
        loss       = money_loss + self.smape_weight * smape_pct

        bs = batch["target"].shape[0]
        self.log("train/money_pct", money_loss, prog_bar=True,  on_step=True, on_epoch=True, batch_size=bs)
        self.log("train/smape",     smape_pct,  prog_bar=True,  on_step=True, on_epoch=True, batch_size=bs)
        self.log("train/loss",      loss,       prog_bar=False, on_step=True, on_epoch=True, batch_size=bs)
        return loss

    def validation_step(self, batch, batch_idx):
        y_true_kwh, y_pred_kwh, dam, sell, buy = self._forward_and_flatten(batch)

        money_loss = money_pct_torch(y_true_kwh, y_pred_kwh, dam, sell, buy)
        smape_pct  = smape_torch(y_true_kwh, y_pred_kwh)

        bs = batch["target"].shape[0]
        self.log("val/money_pct", money_loss, prog_bar=True,  on_epoch=True, on_step=False, batch_size=bs)
        self.log("val/smape",     smape_pct,  prog_bar=True,  on_epoch=True, on_step=False, batch_size=bs)


# ─────────────────────────────────────────────────────────────────────────────
# Per-epoch console print: train_pct=…  train_smape=…  val_pct=…  val_smape=…
# ─────────────────────────────────────────────────────────────────────────────
class EpochMetricsPrinter(L.Callback):
    """Print money_pct / smape after each validation epoch in the requested format."""

    @staticmethod
    def _fetch(metrics, key):
        v = metrics.get(key, None)
        if v is None:
            return None
        try:
            return float(v.item() if hasattr(v, "item") else v)
        except Exception:
            return None

    def on_validation_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics
        tp = self._fetch(m, "train/money_pct_epoch") or self._fetch(m, "train/money_pct")
        ts = self._fetch(m, "train/smape_epoch")     or self._fetch(m, "train/smape")
        vp = self._fetch(m, "val/money_pct")
        vs = self._fetch(m, "val/smape")

        def fmt(x):
            return f"{x:.3f}" if x is not None else "n/a"

        print(
            f"[epoch {trainer.current_epoch:>2d}] "
            f"train_pct={fmt(tp)}% train_smape={fmt(ts)} "
            f"val_pct={fmt(vp)}% val_smape={fmt(vs)}"
        )

## Moirai model initialization (multivariate)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Moirai model initialization — multivariate
# ─────────────────────────────────────────────────────────────────────────────
# Two objects:
#   • MoiraiModule      — the raw transformer weights (loaded from HuggingFace).
#   • MoiraiForecast    — the inference wrapper that produces a GluonTS predictor.
#   • MoiraiFinetune    — the Lightning training wrapper.
# We deepcopy the module into MoiraiFinetune so the zero-shot weights stay
# clean for diagnostic comparisons.
#
# Multivariate dims passed in:
#   target_dim                 = 1
#   feat_dynamic_real_dim      = F  (FUTURE_KNOWN_COLS)
#   past_feat_dynamic_real_dim = P  (PAST_ONLY_COLS)

module = MoiraiModule.from_pretrained(MODEL_ID)

zs_model = MoiraiForecast(
    module=module,
    prediction_length=PRED_LEN,
    target_dim=1,
    feat_dynamic_real_dim=FEAT_DYNAMIC_REAL_DIM,
    past_feat_dynamic_real_dim=PAST_FEAT_DYNAMIC_REAL_DIM,
    context_length=CONTEXT_LEN,
    patch_size=PATCH_SIZE,
    num_samples=NUM_SAMPLES,
)
zs_predictor = zs_model.create_predictor(batch_size=BATCH_SIZE * 2, device=DEVICE)

n_params = sum(p.numel() for p in zs_model.parameters())
print(f"Loaded   : {MODEL_ID}")
print(f"Params   : {n_params:,}  ({n_params / 1e6:.1f} M)")

num_training_steps = len(train_loader) * EPOCHS
num_warmup_steps   = min(500, num_training_steps // 10)

ft_kwargs = dict(
    module=copy.deepcopy(module),
    min_patches=4,
    min_mask_ratio=0.15,
    max_mask_ratio=0.5,
    max_dim=MAX_DIM,                 # accommodate target + covariates
    num_training_steps=num_training_steps,
    num_warmup_steps=num_warmup_steps,
    context_length=CONTEXT_LEN,
    prediction_length=PRED_LEN,
    patch_size=PATCH_SIZE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

ft_model = MoneyLossMoiraiFinetune(
    **ft_kwargs,
    smape_weight=SMAPE_WEIGHT,
    target_n_patches=(CONTEXT_LEN + PRED_LEN) // PATCH_SIZE,
)
monitor_metric = "val/money_pct"
print(f"Training objective: money_pct (per-hour real prices) + {SMAPE_WEIGHT} · SMAPE")
print(f"max_dim          : {MAX_DIM}  (variates per sample: {TOTAL_VARIATES})")

print(f"Trainable params: {sum(p.numel() for p in ft_model.parameters() if p.requires_grad):,}")
print(f"Training steps  : {num_training_steps}  |  warmup: {num_warmup_steps}")

## Training loop + MLflow logging

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Training — Lightning Trainer + MLflow run
# ─────────────────────────────────────────────────────────────────────────────
mlflow.set_experiment("moirai_electricity_forecasting")
mlflow.start_run(run_name=f"moirai-mv-money-ctx{CONTEXT_LEN}-pred{PRED_LEN}")

# ─── Log run-level config / metadata ─────────────────────────────────────────
mlflow.log_params({
    "model_id":                   MODEL_ID,
    "context_len":                CONTEXT_LEN,
    "pred_len":                   PRED_LEN,
    "patch_size":                 PATCH_SIZE,
    "num_samples":                NUM_SAMPLES,
    "train_stride":               TRAIN_STRIDE,
    "batch_size":                 BATCH_SIZE,
    "epochs":                     EPOCHS,
    "lr":                         LR,
    "weight_decay":               WEIGHT_DECAY,
    "loss":                       f"money_pct + {SMAPE_WEIGHT}*smape",
    "y_col":                      Y_COL,
    "group_col":                  GROUP_COL,
    "n_stations":                 len(train_series),
    "feat_dynamic_real_dim":      FEAT_DYNAMIC_REAL_DIM,
    "past_feat_dynamic_real_dim": PAST_FEAT_DYNAMIC_REAL_DIM,
    "total_variates":             TOTAL_VARIATES,
    "max_dim":                    MAX_DIM,
    "train_rows":                 len(train),
    "val_rows":                   len(val),
    "test_rows":                  len(test),
    "train_windows":              len(train_ds),
    "train_date_min":             str(train["datetime"].min()),
    "train_date_max":             str(train["datetime"].max()),
    "val_date_min":               str(val["datetime"].min()),
    "val_date_max":               str(val["datetime"].max()),
    "test_date_min":              str(test["datetime"].min()),
    "test_date_max":              str(test["datetime"].max()),
})
mlflow.log_dict(
    {
        "future_known_cols": FUTURE_KNOWN_COLS,
        "past_only_cols":    PAST_ONLY_COLS,
        "static_cols":       static_cols,
        "calendar_cols":     calendar_cols,
        "cat_columns":       cat_columns,
    },
    "feature_columns.json",
)

# ─── Lightning callbacks ─────────────────────────────────────────────────────
checkpoint_cb = ModelCheckpoint(
    dirpath=CKPT_DIR,
    filename=f"moirai-mv-{{epoch:02d}}-{{{monitor_metric.replace('/', '_')}:.4f}}",
    monitor=monitor_metric,
    mode="min",
    save_top_k=1,
)

trainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision="16-mixed" if torch.cuda.is_available() else "32",
    callbacks=[checkpoint_cb, EpochMetricsPrinter(), TQDMProgressBar(refresh_rate=10)],
    enable_progress_bar=True,
    log_every_n_steps=10,
)

trainer.fit(ft_model, train_loader, val_loss_loader)

print(f"\nBest checkpoint : {checkpoint_cb.best_model_path}")
print(f"Best score      : {checkpoint_cb.best_model_score:.4f}")

# Save fine-tuned weights and log as MLflow artifact
torch.save(ft_model.module.state_dict(), SAVE_PATH)
mlflow.log_artifact(SAVE_PATH, artifact_path="model")
if checkpoint_cb.best_model_path:
    mlflow.log_artifact(checkpoint_cb.best_model_path, artifact_path="checkpoint")
print(f"Fine-tuned weights saved → {SAVE_PATH}")

# Hot-swap weights into the inference predictor
zs_model.module.load_state_dict(ft_model.module.state_dict())
ft_predictor = zs_model.create_predictor(batch_size=BATCH_SIZE * 2, device=DEVICE)
print("Fine-tuned predictor ready")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load fine-tuned multivariate Moirai model (re-run-friendly)
# ─────────────────────────────────────────────────────────────────────────────
# Use this cell to re-create `ft_predictor` from the saved weights without
# rerunning the trainer. The MoiraiForecast wrapper must be constructed with
# the SAME variate dims used during training, otherwise the embedding heads
# won't match.

from gluonts.dataset.common import ListDataset
from tqdm.auto import tqdm
import pandas as pd
import torch
import os

from uni2ts.model.moirai import MoiraiForecast, MoiraiModule

MODEL_PATH = SAVE_PATH

device = "cuda" if torch.cuda.is_available() else "cpu"

module = MoiraiModule.from_pretrained(MODEL_ID)

model = MoiraiForecast(
    module=module,
    prediction_length=PRED_LEN,
    context_length=CONTEXT_LEN,
    patch_size=PATCH_SIZE,
    num_samples=NUM_SAMPLES,
    target_dim=1,
    feat_dynamic_real_dim=FEAT_DYNAMIC_REAL_DIM,
    past_feat_dynamic_real_dim=PAST_FEAT_DYNAMIC_REAL_DIM,
)

state_dict = torch.load(MODEL_PATH, map_location=device)
# strict=False because the saved file is module.state_dict() (raw transformer
# weights); MoiraiForecast wraps the module and adds head buffers.
missing, unexpected = model.module.load_state_dict(state_dict, strict=False)
print(f"Missing keys  : {len(missing)}")
print(f"Unexpected keys: {len(unexpected)}")
model.to(device)
model.eval()

ft_predictor = model.create_predictor(batch_size=256, device=device)

print(f"Loaded multivariate Moirai weights from: {MODEL_PATH}")
print(f"Device: {device}")
print(f"feat_dynamic_real_dim      = {FEAT_DYNAMIC_REAL_DIM}")
print(f"past_feat_dynamic_real_dim = {PAST_FEAT_DYNAMIC_REAL_DIM}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Rolling inference (multivariate) — tile val / test in non-overlapping
# PRED_LEN blocks. Each block is predicted from the preceding CONTEXT_LEN hours
# of GROUND TRUTH target + covariates. This matches the operational setup:
# every PRED_LEN hours we re-run with the latest realised data. (Pure
# recursive prediction would compound error.)
# ─────────────────────────────────────────────────────────────────────────────
# GluonTS conventions for the predictor inputs (uni2ts MoiraiForecast):
#   target                  : 1-D array of length CONTEXT_LEN
#                              (the past target values used as history)
#   feat_dynamic_real       : 2-D array of shape (F, CONTEXT_LEN + PRED_LEN)
#                              (known across context AND prediction horizon)
#   past_feat_dynamic_real  : 2-D array of shape (P_pst, CONTEXT_LEN)
#                              (only the historical portion is supplied)
#   start                   : pd.Period of the FIRST timestamp in target
#   item_id                 : station identifier (string)

def rolling_forecast(predictor, start_cutoff, end_cutoff,
                     split_name="split", infer_batch=256):
    print("── Features used by rolling_forecast ───────────────────────")
    print("Model input variates:")
    print(f"  target           : {Y_COL}  (1)")
    print(f"  feat_dynamic_real (F={FEAT_DYNAMIC_REAL_DIM}):")
    for c in FUTURE_KNOWN_COLS:
        print(f"    • {c}")
    print(f"  past_feat_dynamic_real (P={PAST_FEAT_DYNAMIC_REAL_DIM}):")
    for c in PAST_ONLY_COLS:
        print(f"    • {c}")
    print()
    print("ListDataset fields per window: target, feat_dynamic_real, "
          "past_feat_dynamic_real, start, item_id")
    print("────────────────────────────────────────────────────────────")

    F   = FEAT_DYNAMIC_REAL_DIM
    Ppt = PAST_FEAT_DYNAMIC_REAL_DIM

    all_windows = []
    for eic in tqdm(full_series, desc=f"Collecting {split_name} windows"):
        data        = full_series[eic]
        values      = fill_nan_1d(data["values"])
        feat_future = fill_nan_2d(data["feat_future"])    # (T, F)
        feat_past   = fill_nan_2d(data["feat_past"])      # (T, Ppt)
        tidx        = data["time_idx"]
        dts         = data["datetime"]
        tidx_map    = {int(t): i for i, t in enumerate(tidx)}

        t = start_cutoff + 1
        while t <= end_cutoff:
            ctx_start = t - CONTEXT_LEN
            pred_end  = t + PRED_LEN  # exclusive upper bound for the pred block
            if (ctx_start not in tidx_map) or (t not in tidx_map):
                t += PRED_LEN
                continue
            s, e = tidx_map[ctx_start], tidx_map[t]
            if e - s != CONTEXT_LEN:
                t += PRED_LEN
                continue
            # Need PRED_LEN future hours of feat_dynamic_real available too.
            future_end_tidx = t + PRED_LEN - 1
            if future_end_tidx not in tidx_map:
                t += PRED_LEN
                continue
            fe = tidx_map[future_end_tidx] + 1   # exclusive
            if fe - s != CONTEXT_LEN + PRED_LEN:
                t += PRED_LEN
                continue

            # past target only (CONTEXT_LEN,)
            past_target = values[s:e]
            # full-window future-known covariates: shape (F, CONTEXT_LEN + PRED_LEN)
            fdr = feat_future[s:fe, :].T if F > 0 else None
            # past-only covariates: shape (Ppt, CONTEXT_LEN)
            pfdr = feat_past[s:e, :].T if Ppt > 0 else None

            all_windows.append({
                "eic":      eic,
                "t":        t,
                "ctx":      past_target,
                "fdr":      fdr,
                "pfdr":     pfdr,
                "ts":       values,
                "tidx_map": tidx_map,
                "start":    pd.Period(pd.Timestamp(dts[s]), freq="h"),
            })
            t += PRED_LEN

    records = []
    for b0 in tqdm(range(0, len(all_windows), infer_batch), desc=f"Predicting {split_name}"):
        batch = all_windows[b0 : b0 + infer_batch]

        ds_entries = []
        for w in batch:
            entry = {
                "target":  w["ctx"],
                "start":   w["start"],
                "item_id": w["eic"],
            }
            if F > 0:
                entry["feat_dynamic_real"] = w["fdr"]            # (F, CONTEXT_LEN + PRED_LEN)
            if Ppt > 0:
                entry["past_feat_dynamic_real"] = w["pfdr"]      # (Ppt, CONTEXT_LEN)
            ds_entries.append(entry)

        ds = ListDataset(ds_entries, freq="h", one_dim_target=True)

        forecasts = list(predictor.predict(ds))

        for w, fc in zip(batch, forecasts):
            pred = fc.mean

            for step in range(PRED_LEN):
                ti = w["t"] + step
                if ti not in w["tidx_map"]:
                    break
                pos = w["tidx_map"][ti]
                records.append({
                    GROUP_COL:  w["eic"],
                    "time_idx": int(ti),
                    Y_COL:      float(w["ts"][pos]),
                    "pred":     float(pred[step]),
                })

    return pd.DataFrame(records)


# Run rolling inference on val and test windows
print("── Validation rolling inference ────────────────────────────")
val_pred  = rolling_forecast(ft_predictor, training_cutoff, val_cutoff, split_name="val")
print(f"  val prediction rows : {len(val_pred):,}")

print("── Test rolling inference ──────────────────────────────────")
test_pred = rolling_forecast(ft_predictor, val_cutoff, test_cutoff, split_name="test")
print(f"  test prediction rows: {len(test_pred):,}")

# Re-attach prices and datetime so loss_funcs.money / money_pct work and so the
# existing per-station + plot cells (which expect 'datetime') keep working.
keep = [GROUP_COL, "time_idx", "datetime"] + PRICE_COLS_FOR_LOSS

val_eval  = val_pred.merge(val[keep],  on=[GROUP_COL, "time_idx"], how="inner")
test_eval = test_pred.merge(test[keep], on=[GROUP_COL, "time_idx"], how="inner")

# Sanity: drop any zero-actual rows that would break MAPE/money_pct denominators
val_eval  = val_eval[val_eval[Y_COL].abs() > 1e-6].reset_index(drop=True)
test_eval = test_eval[test_eval[Y_COL].abs() > 1e-6].reset_index(drop=True)

print(f"val_eval  : {val_eval.shape}")
print(f"test_eval : {test_eval.shape}")

# Bias and MAE — added so the existing rolling_eval cell can simply log them.
def _bias(df):
    return float(df["pred"].sum() - df[Y_COL].sum())

def _mae(df):
    return float((df[Y_COL] - df["pred"]).abs().mean())

val_bias_v,  test_bias_v = _bias(val_eval),  _bias(test_eval)
val_mae_v,   test_mae_v  = _mae(val_eval),   _mae(test_eval)
print(f"VAL  bias={val_bias_v:+.2f}  MAE={val_mae_v:.4f}")
print(f"TEST bias={test_bias_v:+.2f}  MAE={test_mae_v:.4f}")

mlflow.log_metrics({
    "val_bias":   val_bias_v,
    "val_mae":    val_mae_v,
    "test_bias":  test_bias_v,
    "test_mae":   test_mae_v,
})

# Persist a small sample of predictions as an MLflow artifact for inspection
sample_path = os.path.join(CKPT_DIR, "test_predictions_sample.csv")
test_eval.head(5000).to_csv(sample_path, index=False)
mlflow.log_artifact(sample_path, artifact_path="predictions")

## Validation / test metrics + MLflow logging

In [ ]:
def _prices(df):
    return df["dam_price"].values, df["sell_bm_price"].values, df["buy_bm_price"].values

val_smape_v     = smape(val_eval[Y_COL], val_eval['pred'])
val_rmse_v      = rmse(val_eval[Y_COL], val_eval['pred'])
val_mape_v      = mape(val_eval[Y_COL], val_eval['pred'])
val_money_v     = money(val_eval[Y_COL], val_eval['pred'], *_prices(val_eval))
val_money_pct_v = money_pct(val_eval[Y_COL], val_eval['pred'], *_prices(val_eval))

test_smape_v     = smape(test_eval[Y_COL], test_eval['pred'])
test_rmse_v      = rmse(test_eval[Y_COL], test_eval['pred'])
test_mape_v      = mape(test_eval[Y_COL], test_eval['pred'])
test_money_v     = money(test_eval[Y_COL], test_eval['pred'], *_prices(test_eval))
test_money_pct_v = money_pct(test_eval[Y_COL], test_eval['pred'], *_prices(test_eval))

print("── Validation ──────────────────────────────────────────────")
print(f"Aligned samples : {len(val_eval):,}")
print(f"SMAPE     : {val_smape_v:.4f}")
print(f"RMSE      : {val_rmse_v:.4f}")
print(f"MAPE      : {val_mape_v:.2f} %")
print(f"MONEY     : {val_money_v:.4f}")
print(f"MONEY_PCT : {val_money_pct_v:.4f}%")

print("── Test ────────────────────────────────────────────────────")
print(f"Aligned samples : {len(test_eval):,}")
print(f"SMAPE     : {test_smape_v:.4f}")
print(f"RMSE      : {test_rmse_v:.4f}")
print(f"MAPE      : {test_mape_v:.2f} %")
print(f"MONEY     : {test_money_v:.4f}")
print(f"MONEY_PCT : {test_money_pct_v:.4f}%")

mlflow.log_metrics({
    "val_smape":      val_smape_v,
    "val_rmse":       val_rmse_v,
    "val_mape":       val_mape_v,
    "val_money":      val_money_v,
    "val_money_pct":  val_money_pct_v,
    "test_smape":     test_smape_v,
    "test_rmse":      test_rmse_v,
    "test_mape":      test_mape_v,
    "test_money":     test_money_v,
    "test_money_pct": test_money_pct_v,
})
mlflow.end_run()
print(f"MLflow run logged → {mlflow.get_tracking_uri()}")

In [ ]:
def per_station_metrics(eval_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for grp, gdf in eval_df.groupby(GROUP_COL):
        rows.append({
            GROUP_COL:    grp,
            "n":          len(gdf),
            "SMAPE":      smape(gdf[Y_COL], gdf["pred"]),
            "RMSE":       rmse (gdf[Y_COL], gdf["pred"]),
            "MAPE":       mape (gdf[Y_COL], gdf["pred"]),
            "MONEY":      money     (gdf[Y_COL], gdf["pred"], *_prices(gdf)),
            "MONEY_PCT":  money_pct (gdf[Y_COL], gdf["pred"], *_prices(gdf)),
        })
    return pd.DataFrame(rows).sort_values("SMAPE")


test_station_metrics = per_station_metrics(test_eval)

print("Top-10 best stations (test SMAPE):")
print(test_station_metrics.head(10).to_string(index=False))
print("\nBottom-10 worst stations (test SMAPE):")
print(test_station_metrics.tail(10).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


def plot_forecast(df, eic_code, start_dt=None, end_dt=None, title_prefix=""):

    df = df[df[GROUP_COL] == eic_code].sort_values("datetime")
    if df.empty:
        raise ValueError(f"No data for EiC code: {eic_code!r}")
    if start_dt is not None:
        df = df[df["datetime"] >= pd.Timestamp(start_dt)]
    if end_dt is not None:
        df = df[df["datetime"] <= pd.Timestamp(end_dt)]
    if df.empty:
        raise ValueError("No data in the specified datetime range.")

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df["datetime"], df[Y_COL],  label="True",      linewidth=1, color="steelblue")
    ax.plot(df["datetime"], df["pred"], label="Predicted", linewidth=1, color="tomato", alpha=0.85)
    ax.set_title(
        f"{title_prefix}{eic_code}  |  MAPE={mape(df[Y_COL], df['pred']):.3f}"
        f"  MONEY_PCT={money_pct(df[Y_COL], df['pred'], *_prices(df)):.2f}"
        f"  ({df['datetime'].min().date()} \u2013 {df['datetime'].max().date()})"
    )
    ax.set_xlabel("Datetime")
    ax.set_ylabel(Y_COL)
    ax.legend()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    fig.autofmt_xdate(rotation=0, ha="center")
    plt.tight_layout()
    plt.show()


# Example:
# plot_forecast(val_eval,  eic_code="<code>")
# plot_forecast(test_eval, eic_code="<code>", start_dt="2025-08-25", end_dt="2025-08-26")

In [ ]:
best_station  = test_station_metrics.iloc[0][GROUP_COL]
worst_station = test_station_metrics.iloc[-1][GROUP_COL]

print(f"Best  station (SMAPE): {best_station}")
plot_forecast(test_eval, eic_code=best_station,  title_prefix="[BEST]  ")

print(f"Worst station (SMAPE): {worst_station}")
plot_forecast(test_eval, eic_code=worst_station, title_prefix="[WORST] ")